In [5]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

gpkg_path = "../data/Kulim_LandUse_2024.gpkg"
layer_name = "GUNATANAH SEMASA KULIM 2024"

output_map = "../image/Kulim_LandUse_2024_Coloured_Map.png"
summary_gtn1_csv = "../data/2024_kulim_landuse_summary.csv"
summary_gtn2_csv = "../data/2024_kulim_landuse_gtn2_summary.csv"

os.makedirs("../image", exist_ok=True)
os.makedirs("../data", exist_ok=True)

gdf = gpd.read_file(gpkg_path, layer=layer_name)

gdf["luas_h"] = pd.to_numeric(gdf["luas_h"], errors="coerce")

summary_gtn1 = (
    gdf.groupby("gtn1", as_index=False)
    .agg(polygons=("geometry", "count"), area_ha=("luas_h", "sum"))
    .sort_values("area_ha", ascending=False)
)
summary_gtn1["area_ha"] = summary_gtn1["area_ha"].round(2)
summary_gtn1.to_csv(summary_gtn1_csv, index=False)

summary_gtn2 = (
    gdf.groupby(["gtn1", "gtn2"], as_index=False)
    .agg(polygons=("geometry", "count"), area_ha=("luas_h", "sum"))
    .sort_values(["gtn1", "area_ha"], ascending=[True, False])
)
summary_gtn2["area_ha"] = summary_gtn2["area_ha"].round(2)
summary_gtn2.to_csv(summary_gtn2_csv, index=False)

colors = {
    "Pertanian": "#93C572",
    "Perumahan": "#FBCEB1",
    "Tanah Kosong": "#6C541E",
    "Pengangkutan": "#FFFF00",
    "Industri": "#880089",
    "Badan Air": "#00FFFF",
    "Hutan": "#228B22",
    "Institusi dan Kemudahan Masyarakat": "#FF0000",
    "Komersial": "#1F75FF",
    "Tanah Lapang dan Rekreasi": "#ADFF2F",
    "Infrastruktur dan Utiliti": "#87CEFA",
}

gdf["plot_color"] = gdf["gtn1"].map(colors).fillna("#DDDDDD")

# Remove empty geometries
gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()

# Repair invalid geometries
invalid_before = (~gdf.geometry.is_valid).sum()
print("Invalid geometries before repair:", invalid_before)

gdf["geometry"] = gdf.geometry.make_valid()

# keep only polygonal geometries after repair
gdf = gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

invalid_after = (~gdf.geometry.is_valid).sum()
print("Invalid geometries after repair:", invalid_after)

# Simplify before dissolve to reduce geometry complexity
gdf["geometry"] = gdf.geometry.simplify(
    tolerance=2,
    preserve_topology=True
)

# Dissolve by major land-use class
gdf_diss = gdf[["gtn1", "plot_color", "geometry"]].dissolve(
    by="gtn1",
    method="unary"
)

# Optional final simplification
gdf_diss["geometry"] = gdf_diss.geometry.simplify(
    tolerance=5,
    preserve_topology=True
)

fig, ax = plt.subplots(figsize=(14, 12))

gdf_diss.plot(
    ax=ax,
    color=gdf_diss["plot_color"],
    edgecolor="black",
    linewidth=0.2
)

legend_items = [
    Patch(facecolor=color, edgecolor="black", label=label)
    for label, color in colors.items()
    if label in set(gdf["gtn1"])
]

ax.legend(
    handles=legend_items,
    title="Land Use Class",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=True,
    fontsize=9,
    title_fontsize=10,
    borderaxespad=0.0
)

ax.set_axis_off()
plt.subplots_adjust(right=0.78)

plt.savefig(output_map, dpi=200, bbox_inches="tight")
plt.close()

print("Saved map:", os.path.abspath(output_map))
print("Saved gtn1 summary:", os.path.abspath(summary_gtn1_csv))
print("Saved gtn2 summary:", os.path.abspath(summary_gtn2_csv))

Invalid geometries before repair: 35
Invalid geometries after repair: 0
Saved map: /Users/yusriyusup/python/analysis_landuse/image/Kulim_LandUse_2024_Coloured_Map.png
Saved gtn1 summary: /Users/yusriyusup/python/analysis_landuse/data/2024_kulim_landuse_summary.csv
Saved gtn2 summary: /Users/yusriyusup/python/analysis_landuse/data/2024_kulim_landuse_gtn2_summary.csv
